# Stage2 improved candidate GPU integration. Public fixtures, bounded install/inference. No competition submission.

In [ ]:
from pathlib import Path
import json,hashlib,zipfile,tempfile,shutil
INPUT=Path('/kaggle/input');WORK=Path(tempfile.mkdtemp(prefix='candidate-check-',dir='/kaggle/working'))
items=list(INPUT.rglob('candidate-assets.json'))
if len(items)!=1:raise ValueError('Expected candidate assets')
asset=items[0].parent;manifest=json.loads(items[0].read_text())
for name,value in manifest.items():
    assert hashlib.sha256((asset/name).read_bytes()).hexdigest()==value
for name,destination in [('candidate.bin',WORK/'candidate'),('public-fixtures.bin',WORK/'fixtures')]:
    with zipfile.ZipFile(asset/name) as z:
        for member in z.namelist():
            assert (destination/member).resolve().is_relative_to(destination.resolve())
        z.extractall(destination)
shutil.copyfile(WORK/'candidate/requirements.txt',WORK/'requirements.txt')
(WORK/'candidate-assets.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
import subprocess, sys, time, os
VENV = Path(tempfile.mkdtemp(prefix='stage3-pinned-', dir='/tmp'))
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
started = time.monotonic()
with (WORK/'install.log').open('w') as log:
    installed = subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--no-cache-dir', '-r', str(WORK/'requirements.txt')], stdout=log, stderr=subprocess.STDOUT,timeout=600)
INSTALL = {'exit_code': installed.returncode, 'seconds': time.monotonic()-started,
           'requirements_sha256': hashlib.sha256((WORK/'requirements.txt').read_bytes()).hexdigest()}
(WORK/'install.json').write_text(json.dumps(INSTALL, indent=2))
if installed.returncode:
    print((WORK/'install.log').read_text()[-12000:])
    raise RuntimeError('Pinned requirements installation failed')
print('Installation:', INSTALL)
RUNNER = "import sys,json,time,socket,importlib.metadata\nfrom pathlib import Path\nimport cv2,torch\nROOT=Path(sys.argv[1]);sys.path.insert(0,str(ROOT/'candidate'))\nimport inference\nversions={}\nfor line in (ROOT/'requirements.txt').read_text().splitlines():\n    if not line.strip() or line.startswith('#'):continue\n    name,expected=line.split('==');actual=importlib.metadata.version(name);assert actual.split('+')[0]==expected;versions[name]=actual\nassert torch.cuda.is_available();torch.set_num_threads(4)\ndef blocked(*args,**kwargs):raise RuntimeError('Network access blocked during inference')\noriginal_connect=socket.socket.connect\ndef guarded_connect(self,address):\n    if self.family in (socket.AF_INET,socket.AF_INET6):return blocked()\n    return original_connect(self,address)\nsocket.socket.connect=guarded_connect;socket.create_connection=blocked\nframes={}\nfor video in sorted((ROOT/'fixtures/stage2/videos').glob('*.mp4')):\n    directory=ROOT/'fixtures/stage2/images'/video.stem;directory.mkdir(parents=True)\n    cap=cv2.VideoCapture(str(video));count=0\n    while True:\n        ok,frame=cap.read()\n        if not ok:break\n        assert cv2.imwrite(str(directory/f'frame_{count:06d}.jpg'),frame);count+=1\n    cap.release();frames[video.stem]=count\nresults={}\nfor stage,columns in [('stage1',['ID','answer']),('stage2',['ID','collision_frame','entry_frame','evasion_space','entry_side']),('stage3',['ID','sample_index','accel_label','steer_label'])]:\n    started=time.monotonic();df=getattr(inference,'predict_'+stage)(ROOT/'fixtures'/stage,ROOT/'candidate/model'/stage)\n    assert list(df.columns)==columns and not df.isna().any().any()\n    if stage=='stage1':assert set(df.ID)=={p.stem for p in (ROOT/'fixtures/stage1/videos').glob('*.mp4')} and not df.ID.duplicated().any() and set(df.answer)<={'ORIGINAL','RERECORDED'}\n    if stage=='stage2':\n        assert set(df.ID)==set(frames) and not df.ID.duplicated().any()\n        assert set(df.evasion_space)<={0,1} and set(df.entry_side)<={'LEFT','RIGHT'}\n        for row in df.itertuples():assert 0<=row.entry_frame<frames[row.ID] and 0<=row.collision_frame<frames[row.ID]\n    if stage=='stage3':\n        assert set(df.ID)=={'OPEN_001'} and len(df)==1200 and df.sample_index.tolist()==list(range(1200))\n        assert set(df.accel_label)<={'ACCELERATING','DECELERATING','CONSTANT','STOPPED'} and set(df.steer_label)<={'LEFT','STRAIGHT','RIGHT'}\n    df.to_csv(ROOT/(stage+'.csv'),index=False);results[stage]=dict(rows=len(df),seconds=time.monotonic()-started)\nreport=dict(status='PASS',scope='Integrated candidate GPU public-example execution; not performance certification',versions=versions,python=sys.version,gpu=torch.cuda.get_device_name(0),python_internet_socket_blocked=True,results=results)\n(ROOT/'integration.json').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n"
(WORK/'integration_runner.py').write_text(RUNNER)
with (WORK/'integration.log').open('w') as log:
    code=subprocess.run([PYTHON,str(WORK/'integration_runner.py'),str(WORK)],stdout=log,stderr=subprocess.STDOUT,timeout=2400).returncode
print((WORK/'integration.log').read_text()[-16000:])
if code:raise RuntimeError('Integration failed')
with zipfile.ZipFile(WORK/'candidate-integration-result.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['integration.json','integration.log','candidate-assets.json','install.json','stage1.csv','stage2.csv','stage3.csv']:
        z.write(WORK/name,name)
